# Lab 11B: The seq2seq bottleneck, and attention

This is the second notebook of Lab 11 (see 11A for an overview).

**The task.** The input is a sequence of random digits, for example `3 8 1 4`, and the model must output
the same digits in **reverse order**: `4 1 8 3`. It is a toy version of translation: a sequence goes in, a
sequence comes out, and every output token depends on one particular input token. Unlike translation, we
know the correct alignment, so we can check whether attention finds it.

| Part | You will… |
|---|---|
| 1. The model | implement attention (three lines from the lecture) and use it in the seq2seq model from the lecture |
| 2. Training and generation | implement autoregressive generation (inference) and teacher forcing (a training trick, explained there) |
| 3. The bottleneck | train a seq2seq model **with** and **without** attention, and compare them on longer and longer inputs |
| 4. What does attention look at? | plot the attention weights |
| 5. Exam-style questions | check your understanding |

## Setup

In [ ]:
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


N_DIGITS = 10                 # the digits 0..9
SOS = 10                      # the "start of sequence" token, fed to the decoder at the first step
VOCAB = N_DIGITS + 1


def make_batch(n, length):
    src = torch.randint(0, N_DIGITS, (n, length), device=device)   # input: random digits
    tgt = src.flip(1)                                              # target: the same digits, reversed
    return src, tgt

---
# Part 1 — The model

### 🛠️ TASK 1: Attention

At each decoding step, the decoder sends a **query** $\mathbf{q}$, compares it with the **key** $\mathbf{k}_i$
of every input position, and retrieves a weighted sum of the **values** $\mathbf{v}_i$ (Lecture 11, Section 5.1):

$$a_i = \frac{\mathbf{q} \cdot \mathbf{k}_i}{\sqrt{d_k}}, \qquad \alpha_i = \text{softmax}(\mathbf{a})_i, \qquad \mathbf{c} = \sum_i \alpha_i \, \mathbf{v}_i$$

(The lecture writes $a_i = \mathbf{q} \cdot \mathbf{k}_i$; the code divides by $\sqrt{d_k}$, the *scaled* dot product,
which keeps the scores in a range where the softmax is not saturated.) With a batch, and possibly several
queries, the same three steps become matrix products:

**🔗 Theory ↔ code**

| Lecture | Code | Shape |
|---|---|---|
| $\mathbf{q} \cdot \mathbf{k}_i$ for all queries and keys: $QK^T$ | `query @ key.transpose(1, 2)` | (batch, n_q, n_k) |
| $\sqrt{d_k}$ | `math.sqrt(d_k)` | |
| $\alpha_i = \text{softmax}(\mathbf{a})_i$, over the keys | `F.softmax(scores, dim=-1)` | (batch, n_q, n_k) |
| $\mathbf{c} = \sum_i \alpha_i \mathbf{v}_i$: $\alpha V$ | `weights @ value` | (batch, n_q, d_v) |

In [ ]:
def attention(query, key, value):
    # query: (batch, n_q, d_k), key: (batch, n_k, d_k), value: (batch, n_k, d_v)
    d_k = query.size(-1)
    scores = ???          # alignment scores a = q . k / sqrt(d_k)    (batch, n_q, n_k)
    weights = ???         # attention weights alpha = softmax(a)      (batch, n_q, n_k), rows sum to 1
    context = ???         # context vector c = sum_i alpha_i v_i      (batch, n_q, d_v)
    return context, weights

In [ ]:
# Test for Task 1
# (1) A query that matches key 2 much better than the others should retrieve value 2
_k = torch.tensor([[[1., 0.], [0., 1.], [-1., 0.], [0., -1.]]])      # 4 keys
_v = torch.tensor([[[1., 1.], [2., 2.], [3., 3.], [4., 4.]]])        # 4 values
_c, _w = attention(torch.tensor([[[-50., 0.]]]), _k, _v)
assert _w.shape == (1, 1, 4) and _c.shape == (1, 1, 2), 'wrong shapes'
assert torch.allclose(_w[0, 0], torch.tensor([0., 0., 1., 0.]), atol=1e-4), 'the query [-50, 0] should put (almost) all weight on key [-1, 0]'
assert torch.allclose(_c[0, 0], torch.tensor([3., 3.]), atol=1e-3), 'the context should be (almost) value 2'
# (2) Random inputs: the weights sum to 1, and the result matches PyTorch's own implementation
_q, _k, _v = torch.randn(2, 3, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 6)
_c, _w = attention(_q, _k, _v)
assert torch.allclose(_w.sum(-1), torch.ones(2, 3), atol=1e-5), 'each row of the weights should sum to 1 (softmax over the keys)'
assert torch.allclose(_c, F.scaled_dot_product_attention(_q, _k, _v), atol=1e-5), 'the result differs from scaled dot-product attention'
print('✅ Task 1 passed')

### The seq2seq model

The model below is `Seq2SeqWithAttention` from **Lecture 11, Code 2**, with one addition: the argument
`use_attention`. With `use_attention=False`, the context vector is replaced by zeros, so the decoder
only knows about the input through the encoder's **final** hidden state, which initialises the decoder:
this is the basic seq2seq model of Section 4.1. Everything else is identical, so any difference between the
two models is caused by attention alone.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, use_attention, vocab_size=VOCAB, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.use_attention = use_attention
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        # Projections for key, value, query
        self.W_k = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_v = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_q = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def encode(self, src):
        enc_out, (h, c) = self.encoder(self.embed(src))     # enc_out: (B, src_len, H): one state per input token
        return enc_out, h, c                                # (h, c): the encoder's FINAL state

    def decode_step(self, tgt_token, h, c, enc_out):
        emb = self.embed(tgt_token)                         # (B, 1, embed_dim)
        if self.use_attention:
            query = self.W_q(h.transpose(0, 1))             # (B, 1, H)  from the decoder's state
            keys = self.W_k(enc_out)                        # (B, src_len, H)  from the encoder's states
            values = self.W_v(enc_out)                      # (B, src_len, H)
            ctx, w = attention(query, keys, values)         # (B, 1, H), (B, 1, src_len)
        else:                                               # basic seq2seq: no context vector
            ctx = torch.zeros(emb.shape[0], 1, h.shape[-1], device=emb.device)
            w = torch.zeros(emb.shape[0], 1, enc_out.shape[1], device=emb.device)
        dec_out, (h, c) = self.decoder(torch.cat([emb, ctx], dim=-1), (h, c))
        logits = self.output(dec_out)                       # (B, 1, vocab)
        return logits, h, c, w

    def forward(self, src, tgt_in):
        # Teacher forcing: at step t the decoder gets tgt_in[:, t] as input
        enc_out, h, c = self.encode(src)
        all_logits, all_weights = [], []
        for t in range(tgt_in.size(1)):
            logits, h, c, w = self.decode_step(tgt_in[:, t:t + 1], h, c, enc_out)
            all_logits.append(logits)
            all_weights.append(w)
        return torch.cat(all_logits, dim=1), torch.cat(all_weights, dim=1)   # (B, tgt_len, vocab), (B, tgt_len, src_len)

---
# Part 2 — Training and generation

The decoder is a **conditional language model**: it predicts the next output token from the input and
the previous output tokens, $P(y_i \mid X, y_1, \ldots, y_{i-1})$. So at every step it needs the
*previous* output token as input. Where does that token come from?

* **Inference (autoregressive generation):** the model generates the output one token at a time, and feeds
  its **own** prediction from the previous step back in as the next input. This is how the model is used
  once it is trained (Lecture 11, Section 4.1).
* **Training:** we could do the same, but early in training the model's own predictions are mostly wrong.
  Every later step would then be trained on a wrong input, and learning would be slow and unstable.
  Instead, we feed the **correct** previous token from the target. This is called **teacher forcing**: like
  a teacher who corrects you after every word, rather than letting you continue from your own mistakes.

> **Teacher forcing in practice.** The decoder input is the target shifted one step to the right, starting
> with the `SOS` ("start of sequence") token. For the target `4 1 8 3` the decoder gets `SOS 4 1 8` and must
> predict `4 1 8 3`: at every step, its input is the correct previous digit.
>
> Teacher forcing is a practical training trick, not part of the exam syllabus. What you need to know is
> that the decoder generates its output **autoregressively**, one token at a time.

### 🛠️ TASK 2: Autoregressive generation and teacher forcing

**🔗 Theory ↔ code**

| Concept | Code |
|---|---|
| the target without its last token: $y_1, \ldots, y_{n-1}$ | `tgt[:, :-1]` |
| put `SOS` in front | `torch.cat([start, ...], dim=1)` |
| the model's own prediction: the most likely token | `logits.argmax(-1)` |

In [ ]:
def decoder_inputs(tgt):
    # Teacher forcing: SOS followed by the target without its last token
    start = torch.full((tgt.shape[0], 1), SOS, device=tgt.device)
    return ???


@torch.no_grad()
def generate(model, src):
    # Autoregressive generation: the decoder gets its OWN previous prediction as input
    enc_out, h, c = model.encode(src)
    token = torch.full((src.shape[0], 1), SOS, device=src.device)    # the first input is SOS
    outputs, weights = [], []
    for t in range(src.shape[1]):                                  # the output is as long as the input
        logits, h, c, w = model.decode_step(token, h, c, enc_out)
        token = ???                                                # the next input: the predicted token
        outputs.append(token)
        weights.append(w)
    return torch.cat(outputs, dim=1), torch.cat(weights, dim=1)

In [ ]:
# Test for Task 2
assert decoder_inputs(torch.tensor([[4, 1, 8, 3]])).tolist() == [[SOS, 4, 1, 8]], 'for the target 4 1 8 3 the decoder input should be SOS 4 1 8'
_pred, _w = generate(Seq2Seq(use_attention=True).to(device), make_batch(5, 7)[0])
assert _pred.shape == (5, 7) and _w.shape == (5, 7, 7), f'generate should return (5, 7) tokens and (5, 7, 7) weights; got {tuple(_pred.shape)}, {tuple(_w.shape)}'
assert _pred.dtype == torch.long, 'the generated tokens should be integers (use argmax)'
print('✅ Task 2 passed')

### The training loop

Each mini-batch has a random length between 5 and 30, so one model learns to reverse sequences of all
these lengths. The loss is the cross-entropy of every output token (as in a language model). Accuracy is
measured with `generate`, so at test time the model must use its own predictions.

In [ ]:
def train_seq2seq(model, steps=2500, lr=1e-3, seed=0):
    set_seed(seed)
    model.to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    t0 = time.time()
    for step in range(1, steps + 1):
        src, tgt = make_batch(128, random.randint(5, 30))
        logits, _ = model(src, decoder_inputs(tgt))
        loss = F.cross_entropy(logits.flatten(0, 1), tgt.flatten())
        optimiser.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)     # avoids rare exploding gradients
        optimiser.step()
        if step % 500 == 0:
            print(f'  step {step}: loss {loss.item():.3f}  ({time.time() - t0:.0f} s)')
    return model


def accuracy(model, length, n=1000):
    # Fraction of correctly generated tokens, overall and for each INPUT position
    src, tgt = make_batch(n, length)
    pred, _ = generate(model, src)
    correct = (pred == tgt).float()                      # (n, length), in output order
    return correct.mean().item(), correct.mean(0).flip(0).cpu()   # flip: output position j holds input position length-1-j

---
# Part 3 — The bottleneck

We train two models, without and with attention. This takes a few minutes (the model with attention is
slower).

✍️ **Prediction:**
1. How does the accuracy of each model change as the input gets longer?
2. For the model without attention, which digits of a long input will it get wrong most often: the first
   ones it read, or the last ones?

In [ ]:
print('Without attention:')
basic = train_seq2seq(Seq2Seq(use_attention=False))
print('With attention:')
attn = train_seq2seq(Seq2Seq(use_attention=True))

In [ ]:
LENGTHS = [5, 10, 15, 20, 25, 30]
models = {'without attention': basic, 'with attention': attn}

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for name, model in models.items():
    acc = [accuracy(model, L)[0] for L in LENGTHS]
    axs[0].plot(LENGTHS, acc, 'o-', label=name)
    print(f'{name:18s}: ' + '   '.join(f'L={L}: {a:.0%}' for L, a in zip(LENGTHS, acc)))
    axs[1].plot(range(1, 26), accuracy(model, 25)[1], label=name)
axs[0].axhline(1 / N_DIGITS, color='grey', ls='--', label='chance')
axs[0].set_xlabel('input length'); axs[0].set_ylabel('fraction of correct digits'); axs[0].set_ylim(0, 1.05)
axs[0].set_title('Accuracy against input length'); axs[0].legend()
axs[1].axhline(1 / N_DIGITS, color='grey', ls='--', label='chance')
axs[1].set_xlabel('position in the INPUT (1 = the first digit read)'); axs[1].set_ylim(0, 1.05)
axs[1].set_title('Inputs of length 25: accuracy per input position'); axs[1].legend()
plt.tight_layout(); plt.show()

### Questions 3

1. How does the accuracy of the model without attention change with the input length? Explain with the
   information bottleneck.
2. Right plot: which input digits does the model without attention get wrong most often? Why those?
3. How does attention change both plots? Why does it help?
4. Even with attention, the accuracy drops somewhat for the longest inputs. Give one reason. (Hint: what
   happens in `generate` after the first mistake?)

---
# Part 4 — What does attention look at?

For one input of length 12, we plot the attention weights $\alpha$ of every decoding step: one row per
output digit, one column per input digit. This is the **alignment matrix** (Lecture 11, Code 2, and the
"Ich hasse diesen Film" figure).

✍️ **Prediction:** where should the bright cells be for the task of reversing a sequence?

In [ ]:
src, tgt = make_batch(1, 12)
pred, weights = generate(attn, src)
plt.figure(figsize=(5, 5))
plt.imshow(weights[0].cpu(), cmap='viridis', vmin=0, vmax=1)
plt.xticks(range(12), src[0].tolist()); plt.yticks(range(12), pred[0].tolist())
plt.xlabel('input digits (source)'); plt.ylabel('generated digits (target)')
plt.title('Attention weights'); plt.colorbar(fraction=0.046); plt.show()
print('input:    ', src[0].tolist())
print('target:   ', tgt[0].tolist())
print('generated:', pred[0].tolist())

### Questions 4

1. Describe the pattern. Is it what you expected for reversing a sequence?
2. What does it mean that each row sums to 1? Why is the context vector called a *soft* selection of the
   input digits, and what does it look like here?
3. Nobody told the model which input digit belongs to which output digit. How did it learn the alignment?
4. Look closely: does each output digit attend to *exactly* the input digit it copies, or to its neighbour?
   (Hint: the encoder state at position $i$ is computed by an LSTM that has read digits $1, \ldots, i$.)

---
# Part 5 — Exam-style questions

**Q1.** In a basic seq2seq model (without attention), what is passed from the encoder to the decoder?

* (a) All hidden states of the encoder.
* (b) The encoder's final hidden state, which initialises the decoder.
* (c) The input tokens themselves.
* (d) The attention weights.

**Q2.** *(Not mandatory: teacher forcing is not part of the exam syllabus.)* During training with teacher forcing, the decoder's
input at step $i$ is

* (a) the decoder's own prediction from step $i - 1$.
* (b) the ground-truth output token $y_{i-1}$.
* (c) the input token $x_i$.
* (d) a vector of zeros.

**Q3.** A basic seq2seq model translates short sentences well but long sentences badly. What is the most
likely explanation?

* (a) The decoder has too many parameters.
* (b) The whole input must be compressed into one fixed-size vector, so information is lost for long inputs.
* (c) The softmax over the vocabulary is too slow.
* (d) The model has no word embeddings.

**Q4.** In the attention mechanism, the attention weights for one decoding step

* (a) are computed from the values only.
* (b) are always one-hot.
* (c) are obtained with a softmax over the alignment scores of all input positions, and sum to 1.
* (d) are fixed before training.